# ihme


In [ ]:
from pyspark.sql import functions as F, DataFrame
from pyspark.sql.types import (
    IntegerType, LongType, DoubleType, BooleanType, StringType, TimestampType,
)

MEDIDAS_VALIDAS  = {"Deaths", "DALYs", "YLDs", "YLLs", "Prevalence", "Incidence"}
METRICAS_VALIDAS = {"Number", "Rate", "Percent"}

# ── Normalización de causa a GBD Nivel 2 (docs/modelo_dimensional.md §6) ───────
# IHME ya trae el nombre GBD Nivel 2 en `causa` -> mapeo directo. Los valores
# fuera de las 16 (totales tipo "All causes" CG0000, o no mapeados) quedan en
# NULL: nunca se fuerzan a una categoría (§6 nota de riesgo).
MAPEO_GBD = {
    "Neoplasms":                                    ("CG0600", 410),
    "Cardiovascular diseases":                      ("CG0530", 491),
    "Neglected tropical diseases and malaria":      ("CG0250", 344),
    "Nutritional deficiencies":                     ("CG0470", 386),
    "Diabetes and kidney diseases":                 ("CG0510", 955),
    "Mental disorders":                             ("CG0610", 545),
    "Neurological disorders":                       ("CG0590", 542),
    "Self-harm and interpersonal violence":         ("CG0860", 717),
    "COVID-19":                                     ("CG0995", 1013),
    "Transport injuries":                           ("CG0810", 688),
    "Digestive diseases":                           ("CG0580", 526),
    "Unintentional injuries":                       ("CG0850", 696),
    "Diarrheal diseases":                           ("CG0350", 302),
    "HIV/AIDS and sexually transmitted infections": ("CG0290", 366),
    "Respiratory infections and tuberculosis":      ("CG0390", 337),
    "Chronic respiratory diseases":                 ("CG0570", 508),
}
_GBD_CODE_MAP     = F.create_map(*[x for k, (c, i) in MAPEO_GBD.items() for x in (F.lit(k), F.lit(c))])
_GBD_CAUSEID_MAP  = F.create_map(*[x for k, (c, i) in MAPEO_GBD.items() for x in (F.lit(k), F.lit(i))])

PAIS_ISO3_MAP = F.create_map(
    F.lit("Guatemala"),   F.lit("GTM"),
    F.lit("Honduras"),    F.lit("HND"),
    F.lit("El Salvador"), F.lit("SLV"),
    F.lit("Costa Rica"),  F.lit("CRI"),
    F.lit("Nicaragua"),   F.lit("NIC"),
    F.lit("Panama"),      F.lit("PAN"),
    F.lit("Panamá"),      F.lit("PAN"),
)

# ── DIM_ETARIO grid canónico (docs/modelo_dimensional.md §5.2) ────────────────
# (id_etario, grupo_edad_codigo, grupo_edad_nombre, edad_min, edad_max, categoria_etaria)
_DIM_ETARIO_GRID = [
    (1,  "LT1",   "Menores de 1",      0, 0,    "Niñez"),
    (2,  "01-04", "1 a 4",             1, 4,    "Niñez"),
    (3,  "05-09", "5 a 9",             5, 9,    "Niñez"),
    (4,  "10-14", "10 a 14",          10, 14,   "Niñez"),
    (5,  "15-19", "15 a 19",          15, 19,   "Juventud"),
    (6,  "20-24", "20 a 24",          20, 24,   "Juventud"),
    (7,  "25-29", "25 a 29",          25, 29,   "Juventud"),
    (8,  "30-34", "30 a 34",          30, 34,   "Adultez"),
    (9,  "35-39", "35 a 39",          35, 39,   "Adultez"),
    (10, "40-44", "40 a 44",          40, 44,   "Adultez"),
    (11, "45-49", "45 a 49",          45, 49,   "Adultez"),
    (12, "50-54", "50 a 54",          50, 54,   "Adultez"),
    (13, "55-59", "55 a 59",          55, 59,   "Adultez"),
    (14, "60-64", "60 a 64",          60, 64,   "Adultez"),
    (15, "65-69", "65 a 69",          65, 69,   "Vejez"),
    (16, "70-74", "70 a 74",          70, 74,   "Vejez"),
    (17, "75-79", "75 a 79",          75, 79,   "Vejez"),
    (18, "80-84", "80 a 84",          80, 84,   "Vejez"),
    (19, "85+",   "85 y más",         85, None, "Vejez"),
    (98, "UNK",   "No especificada",  None, None, "No especificado"),
    (99, "ALL",   "Todas las edades", 0, None, "Total"),
]
# id_etario -> categoria_etaria (para roll-up Niñez/Juventud/Adultez/Vejez/Total)
_ETARIO_CAT_MAP = F.create_map(
    *[x for row in _DIM_ETARIO_GRID for x in (F.lit(row[0]), F.lit(row[5]))]
)

_COLUMN_RENAMES = {
    "population_group": "grupo_poblacion",
    "measure":          "medida",
    "location":         "pais",
    "sex":              "sexo",
    "age":              "grupo_edad",
    "cause":            "causa",
    "metric":           "metrica",
    "year":             "anio",
    "val":              "valor",
    "upper":            "limite_superior",
    "lower":            "limite_inferior",
}

EXPECTED_SCHEMA = {
    "id":              LongType(),
    "grupo_poblacion": StringType(),
    "medida":          StringType(),
    "metrica":         StringType(),
    "pais":            StringType(),
    "pais_iso3":       StringType(),
    "sexo":            StringType(),
    "es_masculino":    BooleanType(),
    "es_femenino":     BooleanType(),
    "es_total":        BooleanType(),
    "es_desconocido":  BooleanType(),
    "grupo_edad":      StringType(),
    "id_etario":       IntegerType(),
    "categoria_etaria":StringType(),
    "causa":           StringType(),
    "gbd_code":        StringType(),
    "gbd_nombre":      StringType(),
    "gbd_cause_id":    IntegerType(),
    "anio":            IntegerType(),
    "valor":           DoubleType(),
    "limite_inferior": DoubleType(),
    "limite_superior": DoubleType(),
    "source_file":     StringType(),
    "ingestion_ts":    TimestampType(),
}


def _verify_schema(df: DataFrame, expected: dict, table_name: str) -> None:
    actual = {f.name: f.dataType for f in df.schema.fields}
    errors = []
    for col, exp_type in expected.items():
        if col not in actual:
            errors.append(f"  MISSING  '{col}'")
        elif type(actual[col]) != type(exp_type):
            errors.append(f"  MISMATCH '{col}': expected {exp_type}, got {actual[col]}")
    extra = [c for c in actual if c not in expected]
    if extra:
        errors.append(f"  EXTRA    {extra}")
    if errors:
        print(f"[SCHEMA] {table_name} — {len(errors)} problema(s):")
        for msg in errors:
            print(msg)
    else:
        print(f"[SCHEMA] {table_name} — OK ({len(actual)} columnas, tipos correctos)")


def _rename_columns(df: DataFrame) -> DataFrame:
    for src, dst in _COLUMN_RENAMES.items():
        if src in df.columns:
            df = df.withColumnRenamed(src, dst)
    return df


def _normalize_sexo(df: DataFrame) -> DataFrame:
    return df.withColumn("sexo",
        F.when(F.col("sexo") == "Both",   F.lit("Ambos"))
         .when(F.col("sexo") == "Male",   F.lit("Hombre"))
         .when(F.col("sexo") == "Female", F.lit("Mujer"))
         .otherwise(F.col("sexo"))
    )


def _normalize_metrica(df: DataFrame) -> DataFrame:
    return df.withColumn("metrica",
        F.when(F.col("metrica") == "Number",  F.lit("Número"))
         .when(F.col("metrica") == "Rate",    F.lit("Tasa"))
         .when(F.col("metrica") == "Percent", F.lit("Porcentaje"))
         .otherwise(F.col("metrica"))
    )


def _cast_numerics(df: DataFrame) -> DataFrame:
    return (
        df
        .withColumn("anio",            F.col("anio").cast(IntegerType()))
        .withColumn("valor",           F.col("valor").cast(DoubleType()))
        .withColumn("limite_superior", F.col("limite_superior").cast(DoubleType()))
        .withColumn("limite_inferior", F.col("limite_inferior").cast(DoubleType()))
    )


def _validate_anio(df: DataFrame) -> DataFrame:
    return df.withColumn("anio",
        F.when(F.col("anio").cast(IntegerType()).between(1990, 2023),
               F.col("anio").cast(IntegerType()))
         .otherwise(F.lit(None).cast(IntegerType()))
    )


def _validate_medida(df: DataFrame) -> DataFrame:
    normalized = F.trim(F.regexp_replace(F.trim(F.col("medida")), r"\s*\(.*\)$", ""))
    return df.withColumn("medida",
        F.when(normalized.isin(*MEDIDAS_VALIDAS), normalized)
         .otherwise(F.lit(None).cast(StringType()))
    )


def _validate_metrica(df: DataFrame) -> DataFrame:
    return df.withColumn("metrica",
        F.when(F.trim(F.col("metrica")).isin("Número", "Tasa", "Porcentaje"),
               F.trim(F.col("metrica")))
         .otherwise(F.lit(None).cast(StringType()))
    )


def _cast_non_negative(df: DataFrame, col: str) -> DataFrame:
    return df.withColumn(col,
        F.when(F.col(col).cast(DoubleType()) >= 0, F.col(col).cast(DoubleType()))
         .otherwise(F.lit(None).cast(DoubleType()))
    )


def _validate_ic_coherence(df: DataFrame) -> DataFrame:
    df = df.withColumn("limite_inferior",
        F.when(
            F.col("limite_inferior").isNotNull()
            & F.col("valor").isNotNull()
            & (F.col("limite_inferior") > F.col("valor")),
            F.lit(None).cast(DoubleType())
        ).otherwise(F.col("limite_inferior"))
    )
    df = df.withColumn("limite_superior",
        F.when(
            F.col("limite_superior").isNotNull()
            & F.col("valor").isNotNull()
            & (F.col("limite_superior") < F.col("valor")),
            F.lit(None).cast(DoubleType())
        ).otherwise(F.col("limite_superior"))
    )
    return df


def _add_sex_booleans(df: DataFrame) -> DataFrame:
    return (
        df
        .withColumn("_sx", F.lower(F.trim(F.col("sexo"))))
        .withColumn("es_masculino",
            F.when(F.col("_sx") == "hombre", F.lit(True)).otherwise(F.lit(False)))
        .withColumn("es_femenino",
            F.when(F.col("_sx") == "mujer",  F.lit(True)).otherwise(F.lit(False)))
        .withColumn("es_total",
            F.when(F.col("_sx") == "ambos",  F.lit(True)).otherwise(F.lit(False)))
        .withColumn("es_desconocido",
            F.when(F.col("_sx").isin("hombre", "mujer", "ambos"), F.lit(False))
             .otherwise(F.lit(True)))
        .drop("_sx")
    )


def _map_pais_iso3(df: DataFrame) -> DataFrame:
    return df.withColumn("pais_iso3", PAIS_ISO3_MAP[F.col("pais")])


def _add_etario_columns(df: DataFrame) -> DataFrame:
    # IHME solo trae 'All ages' (§5.1) -> miembro único ALL (id_etario=99, Total).
    return (
        df
        .withColumn("id_etario",
            F.when(F.col("grupo_edad") == "All ages", F.lit(99))
             .otherwise(F.lit(None).cast(IntegerType())))
        .withColumn("categoria_etaria", _ETARIO_CAT_MAP[F.col("id_etario")])
    )


def _warn_unmapped_countries(df: DataFrame) -> DataFrame:
    unmapped = (
        df.filter(F.col("pais_iso3").isNull())
          .select("pais").distinct()
    )
    count = unmapped.count()
    if count > 0:
        print(f"[WARNING] {count} país/es sin mapeo ISO3 — filas eliminadas:")
        unmapped.show(truncate=False)
    return df.filter(F.col("pais_iso3").isNotNull())


def _add_gbd_columns(df: DataFrame) -> DataFrame:
    # IHME: `causa` ya es el nombre GBD Nivel 2; cie10 no aplica (sin columnas cie10_*).
    code = _GBD_CODE_MAP[F.col("causa")]
    return (
        df
        .withColumn("gbd_code",     code)
        .withColumn("gbd_nombre",
            F.when(code.isNotNull(), F.col("causa")).otherwise(F.lit(None).cast(StringType())))
        .withColumn("gbd_cause_id", _GBD_CAUSEID_MAP[F.col("causa")].cast(IntegerType()))
    )


def transform(df: DataFrame) -> DataFrame:
    df = _rename_columns(df)
    df = _normalize_sexo(df)
    df = _normalize_metrica(df)
    df = _cast_numerics(df)
    df = _validate_anio(df)
    df = _validate_medida(df)
    df = _validate_metrica(df)
    df = _cast_non_negative(df, "valor")
    df = _cast_non_negative(df, "limite_inferior")
    df = _cast_non_negative(df, "limite_superior")
    df = _validate_ic_coherence(df)
    df = _add_sex_booleans(df)
    df = _add_gbd_columns(df)
    df = _add_etario_columns(df)
    df = _map_pais_iso3(df)
    df = _warn_unmapped_countries(df)
    df = (
        df
        .withColumnRenamed("_source_file",         "source_file")
        .withColumnRenamed("_ingestion_timestamp", "ingestion_ts")
        .filter(F.col("valor").isNotNull() & F.col("anio").isNotNull())
        .dropDuplicates()
        .withColumn("id", F.monotonically_increasing_id().cast(LongType()))
    )
    return df.select(
        "id",
        "grupo_poblacion", "medida", "metrica",
        "pais", "pais_iso3",
        "sexo", "es_masculino", "es_femenino", "es_total", "es_desconocido",
        "grupo_edad", "causa",
        "id_etario", "categoria_etaria",
        "gbd_code", "gbd_nombre", "gbd_cause_id",
        "anio",
        "valor", "limite_inferior", "limite_superior",
        "source_file", "ingestion_ts",
    )


spark.sql("CREATE SCHEMA IF NOT EXISTS stage")

try:
    raw    = spark.read.table("semi2.sandbox.raw_ihme")
    staged = transform(raw)
    _verify_schema(staged, EXPECTED_SCHEMA, "stage.ihme")
    display(staged)
    staged.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("stage.ihme")
except Exception as e:
    print(f"Error al procesar sandbox.raw_ihme: {e}")
    raise
